In [1]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 108 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (18.0 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!pip install tqdm pandas ollama scikit-learn

In [5]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(8)
print("Ollama server started")

Ollama server started


In [6]:
!ollama pull llama3.1:8b

In [7]:
!ollama list

NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


In [8]:
import ollama
MODEL_NAME = "llama3.1:8b"

response = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": 'Return only this JSON: {"status": "ok"}'}]
)

print(response["message"]["content"])

{"status": "ok"}


In [1]:
from pathlib import Path
import json
import random
import re
import time


## Synthetic clinical-note generation with **Ollama**

Building the factual profiles and the symptom-enriched profiles were
already run in `02_synthetic_corpus_pipeline_openai.ipynb`. Their clean output —
`clean_symptom_profiles.jsonl` — is the input in this notebook.

We reuse the **exact same prompt, rules, and leakage/quality guards**, and only swap the generation backend to a local Ollama model (`llama3.2:3b`). Nothing in the prompt or the rules is changed.

**Order of cells below:**
1. Config (imports, model name, paths)
2. Load the symptom profiles JSON
3. Diagnostic leakage terms
4. Prompt-safe profile builder + `SYSTEM_PROMPT` + `build_user_prompt` (rules) — verbatim
5. Leakage / quality scanners
6. Ollama call + `generate_clinical_note` (the only backend change)
7. Smoke test (one note) → review before running the full batch
8. Smoke batch (balanced 20 High / 20 Low)
9. Full generation (gated by `RUN_FULL_GENERATION`)

In [2]:
RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)


# --- Folder layout ---
# PROFILES_DIR is the INPUT folder: the cell below reads clean_symptom_profiles.jsonl from here
DATA_DIR = Path("data")
PROFILES_DIR = DATA_DIR / "01_profiles"
NOTES_DIR = DATA_DIR / "02_clinical_notes" / "ollama"
for folder in [PROFILES_DIR, NOTES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CLEAN_JSONL_PATH = PROFILES_DIR / "clean_symptom_profiles.jsonl"
CLINICAL_NOTES_JSONL_PATH = NOTES_DIR / "clinical_notes.jsonl"
SMOKE_TEST_PATH = NOTES_DIR / "smoke_test_note.json"
SMOKE_BATCH_PATH = NOTES_DIR / "smoke_test_batch.json"

print("Ollama corpus output folder:", NOTES_DIR)


Ollama corpus output folder: data\02_clinical_notes\ollama


In [3]:
# Load the clean symptom-enriched profiles produced by Stages A-D (notebook 02).
# This single JSONL file is the only input the corpus generation needs.
# On Colab: if the file is not already present, upload clean_symptom_profiles.jsonl when prompted.
if not CLEAN_JSONL_PATH.exists():
    try:
        from google.colab import files
        print("clean_symptom_profiles.jsonl not found - please upload it now:")
        uploaded = files.upload()
        for fname, content in uploaded.items():
            CLEAN_JSONL_PATH.write_bytes(content)
            print("Saved upload to:", CLEAN_JSONL_PATH)
            break
    except Exception as exc:
        raise FileNotFoundError(
            f"Could not find {CLEAN_JSONL_PATH} and automatic upload failed: {exc}\n"
            "Place clean_symptom_profiles.jsonl in data/profiles/ and re-run this cell."
        )

clean_symptom_profiles = []
with CLEAN_JSONL_PATH.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line:
            clean_symptom_profiles.append(json.loads(line))

from collections import Counter as _Counter
print("Loaded symptom profiles:", len(clean_symptom_profiles))
print("Label balance:", dict(_Counter(sp["label"] for sp in clean_symptom_profiles)))

Loaded symptom profiles: 1856
Label balance: {'High Infection Concern': 824, 'Low Infection Concern': 1032}


In [12]:
# Diagnostic / label-leakage terms. Used by BANNED_NOTE_TERMS below; these must never appear in a generated note.
DIAGNOSIS_LEAKAGE_TERMS = [
    "infection", "infections", "infected",
    "sepsis", "septic", "septicemia", "urosepsis",
    "bacteremia", "bacteraemia",
    "bloodstream infection", "blood stream infection",
    "pneumonia",
    "uti", "urinary tract infection",
    "respiratory infection",
    "cellulitis", "abscess",
    "wound infection", "surgical site infection",
    "meningitis", "endocarditis",
    "osteomyelitis", "empyema",
    "bronchitis", "pyelonephritis",
]

In [18]:
# Prompt-safe profile builder, allowed-column whitelist, writing styles, prompt builders.

# Only these columns may enter the prompt as patient-level structured data.
# Label-related columns must not be included in the clinical note input.
ALLOWED_COLUMNS = {
    "age": "Age",
    "gender": "Sex",
    "admission_type": "Admission type",
    "admission_location": "Admission source",
    "los_days": "Approximate length of stay in days",
    "procedures": "Recorded procedures",
    "WBC_max": "White blood cell count",
    "CRP_max": "CRP",
    "Bands_max": "Band count",
    "Neutrophils_pct_max": "Neutrophil percentage",
    "antibiotics_given": "treatment",
    "antibiotic_routes": "treatment",
}

LABEL_RELATED_COLUMNS = [
    "infection_label",
    "infection_categories",
    "infection_diagnoses",
    "num_infection_dx",
]

# Where each allowed column lives inside the nested factual profile.
ALLOWED_FIELD_LOCATIONS = {
    "age": ("demographics", "age"),
    "gender": ("demographics", "gender"),
    "admission_type": ("admission_context", "admission_type"),
    "admission_location": ("admission_context", "admission_location"),
    "los_days": ("admission_context", "los_days"),
    "procedures": ("procedures_and_support", "procedures"),
    "WBC_max": ("laboratory_values", "wbc_max"),
    "CRP_max": ("laboratory_values", "crp_max"),
    "Bands_max": ("laboratory_values", "bands_max"),
    "Neutrophils_pct_max": ("laboratory_values", "neutrophils_pct_max"),
    "antibiotics_given": ("treatments", "antibiotics_given"),
    "antibiotic_routes": ("treatments", "antibiotic_routes"),
}

# Diagnostic vocabulary that must never appear in a generated note (hard label leakage).
# Antibiotics / treatments are allowed (they are in ALLOWED_COLUMNS).
# Microbiology terms are flagged because they are not provided to the model and would therefore be invented.
BANNED_NOTE_TERMS = DIAGNOSIS_LEAKAGE_TERMS + ["organism", "pathogen", "culture", "cultures"]

WRITING_STYLES = ["Physician progress note", "Nursing handoff note", "Clinical case summary"]
LENGTH_STYLES = ["Short summary", "Medium-length summary"]
NOTE_STRUCTURES = ["Free-text narrative", "Problem-based summary", "Brief handoff-style paragraph"]

# How patient_data fields group, and which groups each profile_focus keeps.
# We keep the demographic anchor plus the fields tied to the primary focus and drop the rest,
# so the model physically cannot write a full inventory of every structured field.
FIELD_GROUPS = {
    "Age": "demographics",
    "Sex": "demographics",
    "Admission type": "admission",
    "Admission source": "admission",
    "Length of stay in days": "admission",
    "Recorded procedures": "procedures",
    "White blood cell count": "labs",
    "CRP": "labs",
    "Band count": "labs",
    "Neutrophil percentage": "labs",
    "Antibiotic exposure": "treatment",
}
# Always kept: the demographic/admission anchor plus LABS. Labs are the discriminative
# signal and exist for BOTH classes -- the VALUES differ, not their mere presence -- so
# they must never be pruned, or High/Low notes become indistinguishable from text.
# Treatment (antibiotics) and procedures are added per profile_focus, which is balanced
# across the corpus (~even split). This keeps the signal while letting antibiotics and
# procedures vary note-to-note, instead of the templated "IV antibiotics were started"
# appearing in nearly every note.
ALWAYS_KEEP_GROUPS = {"demographics", "admission", "labs"}
FOCUS_EXTRA_GROUPS = {
    "admission_overview": [],
    "procedures_and_support": ["procedures"],
    "labs_and_treatments": ["treatment"],
    "treatment_and_course_context": ["treatment", "procedures"],
}
MAX_PROCEDURES_IN_NOTE = 6

# --- Procedure-string normalization -------------------------------------------------
# The raw procedures are ICD-9/10-PCS descriptions that gpt-4o-mini copies verbatim
# (e.g. "for less than 96 consecutive hours", "auxiliary to open heart surgery").
# We normalize them in code so the model never sees the coded phrasing in the first place.

# Explicit rewrites for the most common verbose / coded strings.
PROCEDURE_REWRITES = {
    "continuous invasive mechanical ventilation for less than 96 consecutive hours": "mechanical ventilation",
    "continuous invasive mechanical ventilation for 96 consecutive hours or more": "prolonged mechanical ventilation",
    "non-invasive mechanical ventilation": "non-invasive ventilation",
    "respiratory ventilation, less than 24 consecutive hours": "mechanical ventilation",
    "respiratory ventilation, 24-96 consecutive hours": "mechanical ventilation",
    "respiratory ventilation, greater than 96 consecutive hours": "prolonged mechanical ventilation",

    "central venous catheter placement with guidance": "central venous catheter placement",
    "venous catheterization, not elsewhere classified": "venous catheterization",
    "insertion of endotracheal tube": "endotracheal intubation",
    "insertion of endotracheal airway into trachea": "endotracheal intubation",

    "endoscopic insertion of stent (tube) into bile duct": "endoscopic biliary stent placement",
    "endoscopic insertion of stent (tube) into pancreatic duct": "endoscopic pancreatic stent placement",
    "replacement of stent (tube) in biliary or pancreatic duct": "biliary stent replacement",

    "extracorporeal circulation auxiliary to open heart surgery": "cardiopulmonary bypass",
    "single internal mammary-coronary artery bypass": "internal mammary coronary artery bypass",
    "(aorto)coronary bypass of one coronary artery": "single-vessel coronary artery bypass",
    "(aorto)coronary bypass of two coronary arteries": "two-vessel coronary artery bypass",
    "(aorto)coronary bypass of three coronary arteries": "three-vessel coronary artery bypass",
    "(aorto)coronary bypass of four or more coronary arteries": "multivessel coronary artery bypass",
    "percutaneous transluminal coronary angioplasty [ptca]": "coronary angioplasty",
    "insertion of drug-eluting coronary artery stent(s)": "coronary stent placement",

    "measurement of cardiac sampling and pressure, left heart": "left heart catheterization",
    "measurement of cardiac sampling and pressure, right heart": "right heart catheterization",
    "fluoroscopy of multiple coronary arteries": "coronary angiography",
    "assistance with cardiac output using balloon pump, continuous": "intra-aortic balloon pump support",

    "parenteral infusion of concentrated nutritional substances": "parenteral nutrition",
    "enteral infusion of concentrated nutritional substances": "enteral nutrition",
}

# Coded qualifier phrases to strip wherever they appear (case-insensitive).
_PROC_STRIP_PATTERNS = [
    r",?\s*via natural or artificial opening(?:\s+endoscopic)?",
    r",?\s*percutaneous(?:\s+endoscopic)?\s+approach",
    r",?\s*open approach",
    r",?\s*external approach",
    r",?\s*endoscopic approach",
    r",?\s*posterior approach",
    r",?\s*anterior approach",
    r",?\s*diagnostic\b",
    r"\bauxiliary to open heart surgery\b",
    r"\bfor (?:less than |greater than )?\d+(?:-\d+)? consecutive hours(?: or more)?",
    r"\busing (?:low osmolar |other )?contrast(?:\s+material)?",
    r"\bwith guidance\b",
    r",\s*guidance\b",
    r"\bwith drainage device\b",
    r"\bwith intraluminal device\b",
    r"\bwith autologous (?:venous|arterial) tissue\b",
    r"\bnot elsewhere classified\b",
    r"\bnot otherwise specified\b",
]

_PROC_BRACKET = re.compile(r"\[[^\]]*\]")            # [PTCA], [EGD], [CRT-D]
_PROC_PAREN_CODE = re.compile(r"\((?:aorto|tube)\)", re.IGNORECASE)

# Single-token / fragment garbage from upstream splitting that must never reach a note.
JUNK_PROCEDURES = {
    "femur", "or burn", "infection", "tarsals and metatarsals", "diagnostic",
    "open approach", "percutaneous approach", "posterior approach", "posterior column",
    "multiple", "single", "intermittent", "open", "greater than 96 consecutive hours",
    "less than 6 hours per day", "via natural or artificial opening endoscopic",
    "total system [crt-d]",
}


def normalize_procedure(text):
    """Turn one raw ICD procedure description into natural clinical shorthand."""
    s = str(text).strip()
    key = s.lower()
    if key in PROCEDURE_REWRITES:
        return PROCEDURE_REWRITES[key]
    s = _PROC_BRACKET.sub("", s)
    s = _PROC_PAREN_CODE.sub("", s)
    for pat in _PROC_STRIP_PATTERNS:
        s = re.sub(pat, "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s{2,}", " ", s).strip(" ,;")
    return s


def normalize_procedures(procedures, max_items=None):
    """Clean a list of procedure strings: drop junk fragments, normalize, dedupe, cap."""
    cleaned, seen = [], set()
    for proc in procedures or []:
        if str(proc).strip().lower() in JUNK_PROCEDURES:
            continue
        clean = normalize_procedure(proc)
        if len(clean) < 3:
            continue
        low = clean.lower()
        if low in seen:
            continue
        seen.add(low)
        cleaned.append(clean)
    if max_items is not None:
        cleaned = cleaned[:max_items]
    return cleaned


# --- Lab-value formatting -----------------------------------------------------------
LAB_DISPLAY_NAMES = {"White blood cell count", "CRP", "Band count", "Neutrophil percentage"}


def format_lab_value(value):
    """Drop trailing .0 (4.0 -> 4) and cap genuine decimals at one place (21.55 -> 21.6)."""
    if isinstance(value, bool):
        return value
    if isinstance(value, float):
        return int(value) if value.is_integer() else round(value, 1)
    return value


def sample_writing_style():
    """Sample a writing style, length, and structure combination."""
    return {
        "writing_style": rng.choice(WRITING_STYLES),
        "length_style": rng.choice(LENGTH_STYLES),
        "structure": rng.choice(NOTE_STRUCTURES),
    }


def _generalize_antibiotics(patient_data):
    """Replace specific antibiotic drug names with a generic, route-aware phrase.

    Specific broad-spectrum names (e.g. vancomycin + meropenem) are a near-perfect
    proxy for the infection label, so drug names never reach the prompt; only the
    fact of exposure and the route(s) are kept.
    """
    if "Antibiotics administered" not in patient_data:
        return
    routes = patient_data.pop("Antibiotic routes", None)
    patient_data.pop("Antibiotics administered", None)
    if isinstance(routes, list):
        route_text = " and ".join(str(r) for r in routes if r)
    else:
        route_text = str(routes) if routes else ""
    patient_data["Antibiotic exposure"] = (
        f"{route_text} antibiotics".strip() if route_text else "antibiotics administered"
    )


def build_prompt_safe_profile(symptom_profile):
    """Project a symptom profile onto the ALLOWED_COLUMNS whitelist for the prompt.

    Drops label, infection_source, source_infection_label, microbiology, the
    abnormal-flag columns, and raw identifiers. Missing fields are omitted entirely
    so the model is not tempted to write missing-data statements. Antibiotic drug
    names are generalized, and patient_data is pruned to the profile focus so the
    note emphasizes a few facts instead of listing everything.
    """
    factual = symptom_profile["factual_profile"]

    patient_data = {}
    for col, display_name in ALLOWED_COLUMNS.items():
        section, field = ALLOWED_FIELD_LOCATIONS[col]
        value = factual.get(section, {}).get(field)
        if value is None:
            continue
        if display_name == "Recorded procedures":
            value = normalize_procedures(value, max_items=MAX_PROCEDURES_IN_NOTE)
        elif display_name in LAB_DISPLAY_NAMES:
            value = format_lab_value(value)
        elif display_name == "Length of stay in days" and isinstance(value, (int, float)):
            value = max(1, int(round(value)))
        if isinstance(value, list) and len(value) == 0:
            continue
        patient_data[display_name] = value

    _generalize_antibiotics(patient_data)

    # Always keep the anchor + labs; the focus adds treatment and/or procedures for variety.
    keep_groups = set(ALWAYS_KEEP_GROUPS)
    keep_groups.update(FOCUS_EXTRA_GROUPS.get(factual.get("profile_focus"), ["procedures"]))
    patient_data = {
        name: val
        for name, val in patient_data.items()
        if FIELD_GROUPS.get(name, "other") in keep_groups
    }

    return {
        "profile_focus": factual.get("profile_focus"),
        "secondary_focus": factual.get("secondary_focus"),
        "patient_data": patient_data,
        "symptoms": {
            "selected_symptoms": symptom_profile.get("selected_symptoms", []),
        },
    }


def needs_passive_symptom_voice(safe_profile):
    """True if the patient likely cannot self-report (ventilation/coma/sedation/severe AMS)."""
    procedures = safe_profile["patient_data"].get("Recorded procedures", []) or []
    symptoms = safe_profile.get("symptoms", {}).get("selected_symptoms", []) or []
    haystack = " ".join(str(x).lower() for x in list(procedures) + list(symptoms))
    triggers = ["mechanical ventilation", "ventilator", "intubat", "coma",
                "sedation", "sedated", "altered mental status"]
    return any(t in haystack for t in triggers)


SYSTEM_PROMPT = (
    "You are a clinical documentation assistant generating SYNTHETIC clinical notes "
    "for a de-identified research dataset. You write realistic hospital-style notes "
    "using ONLY the structured information provided. Every clause must trace to a field "
    "in the data. You never diagnose, never name any infection or organism, and never "
    "invent facts, plans, interpretations, or summarizing/monitoring/management sentences "
    "that are not explicitly in the data."
)


def build_user_prompt(safe_profile, style):
    """Build the user prompt for one note."""
    passive_voice = needs_passive_symptom_voice(safe_profile)

    rules = [
            "Use ONLY the patient_data, the selected symptoms, and the requested writing style below.",
            "Emphasize the primary 'profile_focus'. Write a natural note, NOT a complete inventory of the fields; it is fine to leave some details out.",
            "Round and naturalize numbers: say 'about 9 days' for length of stay, and report labs in plain prose (e.g. 'WBC 21.6, neutrophils 86%'). Never use the words maximum, max, or peak.",
            "Paraphrase procedures into natural clinical shorthand (e.g. 'ERCP with biliary stent', 'central line placed', 'intubated and mechanically ventilated'). Never copy coded phrasings such as 'for less than 96 consecutive hours', 'with guidance', or 'continuous invasive'.",
            "If antibiotic exposure is present, refer to it generically and never name a specific drug. Weave it naturally into the clinical course (vary the wording, e.g. 'started on broad antibiotic cover', 'continued on IV antibiotics') rather than appending a stock closing line, and never make it the focus of the note.",
            "Do NOT mention or imply any label, risk level, degree of concern, or symptom-severity category (no/mild/moderate/severe). Describe the listed symptoms naturally.",
            "Do NOT use any of these words or their variants: infection, sepsis, septic, bacteremia, pneumonia, UTI, urinary tract infection, cellulitis, abscess, meningitis, endocarditis, osteomyelitis, empyema, bronchitis, pyelonephritis.",
            "Do NOT name or invent any diagnosis, causative organism, culture / microbiology result, discharge status, or outcome, even if it could be inferred from the data.",
            "Do NOT write missing-data statements such as 'not reported', 'not documented', or 'no data'. If something is missing, simply omit it.",
            "CRITICAL: write ONLY facts present in the data. Do NOT invent or append ANY sentence about monitoring, follow-up, management plans, supportive care, the clinical team's actions, ongoing assessment/evaluation, comprehensive care, or the patient's needs. Banned examples: 'supportive care was provided', 'the clinical team closely monitored', 'as part of his management', 'throughout his stay', 'ongoing assessment', 'tailored to her needs', 'received comprehensive management'. Stop the note the instant the data is exhausted.",
            "Vary sentence structure and wording. Do NOT reuse stock templates such as 'was urgently admitted after being transferred from another hospital', do not open every note with 'The patient is a NN-year-old ...', and do not introduce every lab the same way -- vary it (e.g. 'WBC 14.2', 'WBC of 21.6 with 86% neutrophils', 'white count of 9'), not always 'a white blood cell count of X'.",
            "Do NOT add reasons, causes, justifications, or purposes for any procedure, treatment, or intervention — even if they seem clinically obvious. FORBIDDEN connectors include: 'due to', 'because of', 'secondary to', 'in order to', 'to manage', 'to address', 'for nutritional support', 'for fluid overload', 'necessitated by', 'complicated by', 'as a result of', 'which led to', 'which required'. State each procedure as a standalone fact with no causal link to anything else.",
            "NEVER use these words anywhere in the note: 'elevated', 'abnormal', 'increased', 'decreased', 'marked', 'significant', 'predominance', 'leukocytosis', 'leukopenia', 'neutrophilia', 'thrombocytopenia'. NEVER write phrases like 'WBC is elevated at X', 'CRP is elevated at X', 'a marked increase in', 'significant neutrophil percentage', 'with a low band count'. Replace EVERY such phrase with the raw number alone, even when the value is clearly abnormal. EXAMPLES: WRONG → 'WBC is elevated at 36.8' || CORRECT → 'WBC is 36.8'. WRONG → 'marked increase in neutrophils at 92.6%' || CORRECT → 'neutrophils 92.6%'. WRONG → 'a band count of 0 is low' || CORRECT → 'band count 0'. This rule applies regardless of whether the value is clinically abnormal.",
            ]

    if passive_voice:
        rules.append(
            "The patient may be unable to self-report (ventilation, sedation, coma, or severe "
            "altered mental status). Do NOT write that the patient 'reported' symptoms; instead "
            "use wording like 'documented', 'observed', or 'was noted to have'."
        )

    rules_text = "\n".join(f"- {r}" for r in rules)

    prompt = (
        f"Write ONE synthetic clinical note.\n\n"
        f"Writing style: {style['writing_style']}\n"
        f"Length: {style['length_style']}\n"
        f"Structure: {style['structure']}\n\n"
        f"Rules:\n{rules_text}\n\n"
        f"Patient data and symptoms (JSON):\n"
        f"{json.dumps(safe_profile, ensure_ascii=False, indent=2)}\n\n"
        f'Return JSON only, in exactly this form: {{"clinical_note": "..."}}'
    )
    return prompt


print("Stage E prompt builders ready.")
print("Allowed columns:", len(ALLOWED_COLUMNS))
print("Label-related columns excluded from patient data:", LABEL_RELATED_COLUMNS)
print("Writing styles:", WRITING_STYLES)

Stage E prompt builders ready.
Allowed columns: 12
Label-related columns excluded from patient data: ['infection_label', 'infection_categories', 'infection_diagnoses', 'num_infection_dx']
Writing styles: ['Physician progress note', 'Nursing handoff note', 'Clinical case summary']


In [19]:
# Post-hoc leakage / quality scanners (verbatim from the OpenAI pipeline).

_banned_note_pattern = re.compile(
    r"\b(" + "|".join(re.escape(t) for t in sorted(set(BANNED_NOTE_TERMS), key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)


def scan_banned_terms(text):
    """Return banned diagnostic terms found in a generated note."""
    if not text:
        return []
    return sorted(set(m.lower() for m in _banned_note_pattern.findall(text)))

SEVERITY_LEAKAGE_TERMS = [
    "mild symptoms",
    "moderate symptoms",
    "severe symptoms",
    "no symptoms",
    "symptom severity",
]


def scan_severity_leakage(text):
    """Return symptom-severity leakage phrases found in a generated note."""
    if not text:
        return []
    lower = text.lower()
    return [term for term in SEVERITY_LEAKAGE_TERMS if term in lower]


def scan_treatment_shortcuts(text):
    lower = text.lower()
    terms = []
    if "antibiotics" in lower:
        terms.append("antibiotics")
    if "a total of" in lower:
        terms.append("a total of")
    if "vancomycin" in lower:
        terms.append("vancomycin")
    if "meropenem" in lower:
        terms.append("meropenem")
    return terms


_BOILERPLATE_PATTERNS = [
    r"supportive care", r"management plan", r"as part of (his|her|the) management",
    r"clinical team", r"clinical management", r"ongoing (assessment|evaluation|management)",
    r"closely monitored", r"monitored closely", r"monitoring of the patient",
    r"comprehensive (care|management)", r"supportive measures", r"as needed",
    r"tailored to", r"treatment response", r"further care", r"appropriate management",
    r"evolving needs", r"continued supportive",
    r"throughout (his|her) (stay|hospitalization|admission|course)",
    r"to address (his|her) (clinical )?(presentation|needs|condition)",

    # Ollama hallucination / interpretation patterns
    r"\bstable\b",
    r"remained stable",
    r"has been stable",
    r"vital signs",
    r"closely monitored",
    r"close monitoring",
    r"\bicu\b",
    r"under observation",
    r"awaiting further evaluation",
    r"\bdue to\b",
    r"\bbecause of\b",
    r"for suspected",
    r"\bsuspected\b",
    r"\byesterday\b",
    r"on the second day",
    r"since admission",
    r"since arrival",
    r"respiratory distress",
    r"for a variety of indications",
    r"following the procedures",

    # Ollama factuality / prompt-leakage warnings
    r"primary focus",
    r"profile_focus",
    r"labs and treatments",
    r"procedures and support",
    r"admission overview",
    r"treatment and course context",

    # Missing-data statements
    r"no selected symptoms",
    r"no selected symptoms are documented",
    r"no symptoms are documented",
    r"no symptoms documented",

    # Course / timing / causality phrases
    r"\bdue to\b",
    r"due to respiratory distress",
    r"since admission",
    r"under observation",
    r"under observation for about",
    r"has been hospitalized",
    r"hospitalized for about",
    r"stayed in the hospital",
    r"has been admitted for about",
    r"admitted for approximately",
]

def scan_boilerplate(text):
    """Detect boilerplate, interpretation, or likely hallucination phrases."""
    lower = text.lower()
    found = set()

    for pattern in _BOILERPLATE_PATTERNS:
        match = re.search(pattern, lower)
        if match:
            found.add(match.group(0))

    return sorted(found)


print("Leakage / quality scanners ready.")

Leakage / quality scanners ready.


In [20]:
# Ollama generation. This is the ONLY backend change vs. the OpenAI notebook:
# call_openai_json -> call_ollama_json. The prompt, rules, and guards are identical.

def call_ollama_json(system_prompt, user_prompt, max_retries=3):
    """Call the local Ollama model in JSON mode, with simple retries."""
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = ollama.chat(
                model=MODEL_NAME,
                format="json",
                options={"temperature": 0.3},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            return response["message"]["content"]
        except Exception as exc:  # transient / model-load errors
            last_error = exc
            wait = 2 ** attempt
            print(f"  Ollama call failed (attempt {attempt}/{max_retries}): {exc}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"Ollama call failed after {max_retries} attempts: {last_error}")


def generate_clinical_note(symptom_profile, note_id, style=None):
    """Generate one clinical note record from a symptom-enriched profile."""
    if style is None:
        style = sample_writing_style()

    safe_profile = build_prompt_safe_profile(symptom_profile)
    user_prompt = build_user_prompt(safe_profile, style)

    raw = call_ollama_json(SYSTEM_PROMPT, user_prompt)

    try:
        parsed = json.loads(raw)
        clinical_note = str(parsed["clinical_note"]).strip()
    except (json.JSONDecodeError, KeyError, TypeError) as exc:
        raise ValueError(f"Model did not return valid {{'clinical_note': ...}} JSON: {exc}\nRaw: {raw[:500]}")

    leaked_terms = scan_banned_terms(clinical_note)
    severity_leakage_terms = scan_severity_leakage(clinical_note)
    treatment_shortcut_terms = scan_treatment_shortcuts(clinical_note)
    boilerplate_terms = scan_boilerplate(clinical_note)

    # Re-attach internal metadata in code (the model never saw these).
    record = {
        "note_id": str(note_id),
        "clinical_note": clinical_note,
        "label": symptom_profile["label"],
        "infection_source": symptom_profile["infection_source"],
        "source_infection_label": symptom_profile["source_infection_label"],
        "source_symptom_profile_id": symptom_profile["symptom_profile_id"],
        "source_profile_id": symptom_profile["source_profile_id"],
        "source_subject_id": symptom_profile["source_subject_id"],
        "source_hadm_id": symptom_profile["source_hadm_id"],
        "symptom_severity": symptom_profile["symptom_severity"],
        "selected_symptoms": symptom_profile["selected_symptoms"],
        "writing_style": style["writing_style"],
        "length_style": style["length_style"],
        "structure": style["structure"],
        "model": MODEL_NAME,
        "leaked_terms": leaked_terms,
        "severity_leakage_terms": severity_leakage_terms,
        "treatment_shortcut_terms": treatment_shortcut_terms,
        "boilerplate_terms": boilerplate_terms,
        "note_char_length": len(clinical_note),
    }
    return record


print("generate_clinical_note ready (Ollama backend).")

generate_clinical_note ready (Ollama backend).


In [21]:
# Smoke test: generate ONE clinical note and review it.

smoke_profile = clean_symptom_profiles[0]

print("Prompt-safe profile sent to the model:")
print(json.dumps(build_prompt_safe_profile(smoke_profile), ensure_ascii=False, indent=2))
print("\n(Internal label, kept out of the prompt:", smoke_profile["label"], ")")
print("=" * 80)

smoke_note = generate_clinical_note(smoke_profile, note_id="smoke_0001")

print("\nGenerated clinical note:")
print("-" * 80)
print(smoke_note["clinical_note"])
print("-" * 80)
print("writing_style:", smoke_note["writing_style"])
print("length_style:", smoke_note["length_style"])
print("structure:", smoke_note["structure"])
print("symptom_severity:", smoke_note["symptom_severity"])
print("selected_symptoms:", smoke_note["selected_symptoms"])
print("note_char_length:", smoke_note["note_char_length"])
print("leaked_terms:", smoke_note["leaked_terms"])
print("severity_leakage_terms:", smoke_note["severity_leakage_terms"])

if smoke_note["leaked_terms"] or smoke_note["severity_leakage_terms"]:
    print("\nWARNING: leakage-like terms found in the note. Review the prompt before full generation.")
else:
    print("\nNo banned or severity-leakage terms detected.")

with SMOKE_TEST_PATH.open("w", encoding="utf-8") as file:
    json.dump(smoke_note, file, ensure_ascii=False, indent=2)

print("\nSmoke-test note saved to:", SMOKE_TEST_PATH)

Prompt-safe profile sent to the model:
{
  "profile_focus": "admission_overview",
  "secondary_focus": "procedures_and_support",
  "patient_data": {
    "Age": 47,
    "Sex": "M",
    "Admission type": "urgent admission",
    "Admission source": "transferred from another hospital",
    "White blood cell count": 21.6,
    "Band count": 4,
    "Neutrophil percentage": 86
  },
  "symptoms": {
    "selected_symptoms": [
      "chills",
      "fatigue",
      "fever",
      "malaise"
    ]
  }
}

(Internal label, kept out of the prompt: High Infection Concern )

Generated clinical note:
--------------------------------------------------------------------------------
The patient was urgently admitted after being transferred from another hospital. He has been experiencing chills and fatigue about his admission. His white blood cell count is 21.6 with 86% neutrophils, band count 4. The patient also reports fever and malaise.
---------------------------------------------------------------------

In [22]:
# Smoke batch: balanced 20 High / 20 Low from DISTINCT patients.
# Writes to smoke_test_batch.json; does NOT touch the full clinical_notes.jsonl run.
import pandas as pd

N_PER_LABEL = 20

# Sample N_PER_LABEL profiles for EACH label (balanced 20 High / 20 Low), stratified across
# profile_focus and drawn from distinct patients. We bucket per (label, focus) and round-robin
# across the focus buckets, skipping already-used patients.
from collections import defaultdict, deque
_by_label_focus = defaultdict(lambda: defaultdict(deque))
for sp in clean_symptom_profiles:
    _by_label_focus[sp["label"]][sp["factual_profile"].get("profile_focus")].append(sp)

seen_subjects = set()


def _take_balanced(label, n):
    buckets = list(_by_label_focus[label].values())
    picked = []
    while buckets and len(picked) < n:
        for b in buckets:
            while b:  # pop until an unseen patient (or bucket empties)
                cand = b.popleft()
                if cand.get("source_subject_id") in seen_subjects:
                    continue
                seen_subjects.add(cand.get("source_subject_id"))
                picked.append(cand)
                break
            if len(picked) >= n:
                break
        buckets = [b for b in buckets if b]
    return picked


_high = _take_balanced("High Infection Concern", N_PER_LABEL)
_low = _take_balanced("Low Infection Concern", N_PER_LABEL)
# Interleave High/Low so the printed review alternates labels.
smoke_batch_profiles = [p for pair in zip(_high, _low) for p in pair]
smoke_batch_profiles += _high[len(_low):] + _low[len(_high):]
print(f"Sampled {len(_high)} High + {len(_low)} Low from {len(seen_subjects)} distinct patients.")

smoke_batch_notes = []
total = len(smoke_batch_profiles)
for i, sp in enumerate(smoke_batch_profiles, start=1):
    print(f"Generating {i}/{total}...")
    record = generate_clinical_note(sp, note_id=f"smoke_batch_{i:04d}")
    smoke_batch_notes.append(record)
    print(
        f"Done {i}/{total} | "
        f"chars={record['note_char_length']} | "
        f"leak={len(record['leaked_terms'])} | "
        f"sev={len(record['severity_leakage_terms'])} | "
        f"boiler={len(record['boilerplate_terms'])}"
    )

# Per-note quality summary (focus from the profile, flags from the record).
summary_rows = []
for sp, rec in zip(smoke_batch_profiles, smoke_batch_notes):
    summary_rows.append({
        "note_id": rec["note_id"],
        "subject": rec["source_subject_id"],
        "label": rec["label"],
        "profile_focus": sp["factual_profile"].get("profile_focus"),
        "writing_style": rec["writing_style"],
        "chars": rec["note_char_length"],
        "leaked": rec["leaked_terms"],
        "severity_leak": rec["severity_leakage_terms"],
        "treatment_shortcut": rec["treatment_shortcut_terms"],
        "boilerplate": rec["boilerplate_terms"],
    })

smoke_batch_summary_df = pd.DataFrame(summary_rows)
print("Generated", len(smoke_batch_notes), "notes from",
      len(seen_subjects), "distinct patients.\n")
display(smoke_batch_summary_df)

n_leak = sum(1 for r in smoke_batch_notes if r["leaked_terms"])
n_sev = sum(1 for r in smoke_batch_notes if r["severity_leakage_terms"])
n_treat = sum(1 for r in smoke_batch_notes if r["treatment_shortcut_terms"])
n_boiler = sum(1 for r in smoke_batch_notes if r["boilerplate_terms"])
print(f"\nNotes with banned/diagnosis terms : {n_leak}/{len(smoke_batch_notes)}")
print(f"Notes with severity leakage       : {n_sev}/{len(smoke_batch_notes)}")
print(f"Notes with treatment-shortcut term: {n_treat}/{len(smoke_batch_notes)}")
print(f"Notes with invented boilerplate    : {n_boiler}/{len(smoke_batch_notes)}")
from collections import Counter as _Counter
print("Label balance:", dict(_Counter(r["label"] for r in smoke_batch_notes)))

print("\n" + "=" * 80)
for rec in smoke_batch_notes:
    print(f"\n[{rec['note_id']}] {rec['label']} | {rec['writing_style']} | {rec['length_style']}")
    print("-" * 80)
    print(rec["clinical_note"])

with SMOKE_BATCH_PATH.open("w", encoding="utf-8") as file:
    json.dump(smoke_batch_notes, file, ensure_ascii=False, indent=2)

print("\nSmoke-batch saved to:", SMOKE_BATCH_PATH)

Sampled 20 High + 20 Low from 40 distinct patients.
Generating 1/40...
Done 1/40 | chars=292 | leak=0 | sev=0 | boiler=0
Generating 2/40...
Done 2/40 | chars=177 | leak=0 | sev=0 | boiler=0
Generating 3/40...
Done 3/40 | chars=372 | leak=0 | sev=0 | boiler=0
Generating 4/40...
Done 4/40 | chars=302 | leak=0 | sev=0 | boiler=0
Generating 5/40...
Done 5/40 | chars=276 | leak=0 | sev=0 | boiler=0
Generating 6/40...
Done 6/40 | chars=202 | leak=0 | sev=0 | boiler=0
Generating 7/40...
Done 7/40 | chars=583 | leak=0 | sev=0 | boiler=0
Generating 8/40...
Done 8/40 | chars=187 | leak=0 | sev=0 | boiler=0
Generating 9/40...
Done 9/40 | chars=233 | leak=0 | sev=0 | boiler=0
Generating 10/40...
Done 10/40 | chars=497 | leak=0 | sev=0 | boiler=0
Generating 11/40...
Done 11/40 | chars=342 | leak=0 | sev=0 | boiler=0
Generating 12/40...
Done 12/40 | chars=251 | leak=0 | sev=0 | boiler=0
Generating 13/40...
Done 13/40 | chars=293 | leak=0 | sev=0 | boiler=0
Generating 14/40...
Done 14/40 | chars=200 

,note_id,subject,label,profile_focus,writing_style,chars,leaked,severity_leak,treatment_shortcut,boilerplate
0,smoke_batch_0001,10004235,High Infection Concern,admission_overview,Clinical case summary,292,[],[],[],[]
1,smoke_batch_0002,10009628,Low Infection Concern,admission_overview,Nursing handoff note,177,[],[],[],[]
2,smoke_batch_0003,10018081,High Infection Concern,procedures_and_support,Nursing handoff note,372,[],[],[],[]
3,smoke_batch_0004,10006053,Low Infection Concern,procedures_and_support,Nursing handoff note,302,[],[],[],[]
4,smoke_batch_0005,10005817,High Infection Concern,labs_and_treatments,Clinical case summary,276,[],[],[],[]
5,smoke_batch_0006,10031404,Low Infection Concern,labs_and_treatments,Clinical case summary,202,[],[],[],[]
6,smoke_batch_0007,10002495,High Infection Concern,treatment_and_course_context,Nursing handoff note,583,[],[],[],[]
7,smoke_batch_0008,10019385,Low Infection Concern,admission_overview,Nursing handoff note,187,[],[],[],[]
8,smoke_batch_0009,10037861,High Infection Concern,admission_overview,Physician progress note,233,[],[],[],[]
9,smoke_batch_0010,10038081,Low Infection Concern,procedures_and_support,Physician progress note,497,[],[],[],[]



Notes with banned/diagnosis terms : 0/40
Notes with severity leakage       : 0/40
Notes with treatment-shortcut term: 4/40
Notes with invented boilerplate    : 3/40
Label balance: {'High Infection Concern': 20, 'Low Infection Concern': 20}


[smoke_batch_0001] High Infection Concern | Clinical case summary | Short summary
--------------------------------------------------------------------------------
A 47-year-old male was urgently admitted after being transferred from another hospital. He presented with chills, fatigue, fever, and malaise about 9 days prior to admission. His white blood cell count is 21.6 with 86% neutrophils and a band count of 4. ERCP with biliary stent was performed.

[smoke_batch_0002] Low Infection Concern | Nursing handoff note | Short summary
--------------------------------------------------------------------------------
The patient was urgently admitted after being transferred from another hospital. He has been experiencing chills and fatigue about his admi

In [23]:
# Full generation (Ollama). Disabled by default - flip the flag after the smoke test.

RUN_FULL_GENERATION = True  # Set to True to generate the full corpus of notes. Runs locally on the GPU.
NOTES_PER_PROFILE = 1         # 1 or 2 notes per symptom profile
MAX_NOTES = None              # e.g. 50 for a small trial run; None = no cap

# Output file for the full corpus (alias to the path defined in the Stage E setup cell).
NOTES_JSONL_PATH = CLINICAL_NOTES_JSONL_PATH

if not RUN_FULL_GENERATION:
    print("Full generation is OFF. Set RUN_FULL_GENERATION = True to generate the corpus.")
else:
    # Resume support: collect note_ids already written.
    already_done = set()
    if NOTES_JSONL_PATH.exists():
        with NOTES_JSONL_PATH.open("r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()
                if line:
                    try:
                        already_done.add(json.loads(line)["note_id"])
                    except Exception:
                        pass
    print("Notes already generated (resume):", len(already_done))

    generated = 0
    failed = 0
    leak_flagged = 0

    with NOTES_JSONL_PATH.open("a", encoding="utf-8") as out_file:
        for sp in clean_symptom_profiles:
            for k in range(1, NOTES_PER_PROFILE + 1):
                note_id = f"{sp['symptom_profile_id']}_n{k}"

                if note_id in already_done:
                    continue
                if MAX_NOTES is not None and generated >= MAX_NOTES:
                    break

                try:
                    record = generate_clinical_note(sp, note_id=note_id)
                except Exception as exc:
                    failed += 1
                    print(f"  Failed note {note_id}: {exc}")
                    continue

                out_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                out_file.flush()
                generated += 1
                if record["leaked_terms"]:
                    leak_flagged += 1

                if generated % 25 == 0:
                    print(f"  Generated {generated} notes...")

            if MAX_NOTES is not None and generated >= MAX_NOTES:
                print(f"Reached MAX_NOTES = {MAX_NOTES}.")
                break

    print("\nGeneration finished.")
    print("Newly generated:", generated)
    print("Failed:", failed)
    print("Notes flagged with leaked terms:", leak_flagged)
    print("Output file:", NOTES_JSONL_PATH)

Notes already generated (resume): 0
  Generated 25 notes...
  Generated 50 notes...
  Generated 75 notes...
  Generated 100 notes...
  Generated 125 notes...
  Generated 150 notes...
  Generated 175 notes...
  Generated 200 notes...
  Generated 225 notes...
  Generated 250 notes...
  Generated 275 notes...
  Generated 300 notes...
  Generated 325 notes...
  Generated 350 notes...
  Generated 375 notes...
  Generated 400 notes...
  Generated 425 notes...
  Generated 450 notes...
  Generated 475 notes...
  Generated 500 notes...
  Generated 525 notes...
  Generated 550 notes...
  Generated 575 notes...
  Generated 600 notes...
  Generated 625 notes...
  Generated 650 notes...
  Generated 675 notes...
  Generated 700 notes...
  Generated 725 notes...
  Generated 750 notes...
  Generated 775 notes...
  Generated 800 notes...
  Generated 825 notes...
  Generated 850 notes...
  Generated 875 notes...
  Generated 900 notes...
  Generated 925 notes...
  Generated 950 notes...
  Generated 975 n